In [ ]:
conda create -n evo2_c python=3.11

#### 安装包含子模块的仓库、验证子模块是否安装成功

In [ ]:
# 进入主仓库目录，运行：
git submodule status --recursive
# 若输出中 没有 - 或 + 符号（如所有子模块显示完整哈希），则所有模块已成功拉取。

In [ ]:
# 如果子模块失败
# 以 cudnn-frontend 为例，手动更新子模块（3rdparty/cudnn-frontend这种具体路径要在代码的输出里寻找，找到正确的）：
git submodule update --init --recursive 3rdparty/cudnn-frontend

#### 查看路径

In [ ]:
#（查看pip路径）
type -a pip 
pip --version
#（设置当前的第一路径）
export PATH="/home/huazhi/anaconda3/envs/evo2c/bin:$PATH" 


which python
# 该行代码只是一次性的，之后每次进入该虚拟环境都需要重新设置
export PYTHONPATH="/home/huazhi/anaconda3/envs/evo2c/bin/python:$PYTHONPATH"

#### CUDA、cuDNN、GCC：
安装 CUDA（特别是 CUDA Toolkit）时需要考虑 GCC（GNU Compiler Collection）的兼容性。<br>
这是因为 nvcc（NVIDIA CUDA Compiler）在编译 Host Code（CPU部分代码）时会调用系统的 GCC（或 Clang），<br>
而不同版本的 nvcc（对应不同 CUDA）对 GCC 最高支持的版本有限制。<br>
CUDA Toolkit（CUDA的运行版本） 是通过 nvcc --version 来判断版本，和 CUDA的驱动版本（通过nvidia-smi查看） 不一样 <br>
cuDNN需直接安装到对应版本的CUDA目录（如/usr/local/cuda-11.3）。<br>
若涉及编译（如CUDA加速），需确保已安装CUDA Toolkit、cuDNN等系统级依赖

#### CUDA安装

In [ ]:
# 查看所有CUDA安装：
ls -l /usr/local | grep cuda
# 查看环境里所安装的cuda
ls -l /usr/local | grep -E 'cuda|cuda-[0-9]+\.[0-9]+'

# 安装CUDA
wget https://developer.download.nvidia.com/compute/cuda/12.4.0/local_installers/cuda_12.4.0_550.54.14_linux.run
sudo sh cuda_12.4.0_550.54.14_linux.run
# 安装的时候，如果有 driver，就只取消安装 driver 就行，其他的都安装即可

In [ ]:
wget https://developer.download.nvidia.com/compute/cuda/12.4.1/local_installers/cuda-repo-rhel7-12-4-local-12.4.1_550.54.15-1.x86_64.rpm
sudo rpm -i cuda-repo-rhel7-12-4-local-12.4.1_550.54.15-1.x86_64.rpm
sudo yum clean all
sudo yum -y install cuda-toolkit-12-4

检查CUDA核心文件是否存在：

In [ ]:
ls /usr/local/cuda-12.4/bin/nvcc  # 验证nvcc编译器是否存在
ls /usr/local/cuda-12.4/lib64     # 验证CUDA库文件是否存在

##### CUDA环境变量配置

直接在终端命令里输入命令：

In [ ]:
export CUDA_HOME=/usr/local/cuda-12.4
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH
ls $CUDA_HOME/include/cuda_runtime.h
which nvcc  # 应输出/usr/local/cuda-12.4/bin/nvcc
ls $CUDA_HOME/lib64/libcudart.so

通过修改 .bashrc 文件 <br>

|场景|	                操作 |
|  ----  | ----  |
|打开/创建文件  |  sudo vim \~/.bashrc（已存在则打开，不存在则创建，但需注意权限） |
|编辑内容	    |     按 i 进入插入模式 → 修改 → Esc → :wq 保存退出 |
|修复权限	    |    sudo chown $USER:$USER \~/.bashrc |
|生效配置	    |    source \~/.bashrc 或重启终端|

In [ ]:
# 打开 .bashrc 文件
sudo vim \~/.bashrc
# 在文件最后添加，以设置CUDA路径
export PATH=/usr/local/cuda-12.4/bin${PATH:+:${PATH}}
export LD_LIBRARY_PATH=/usr/local/cuda-12.4/lib64${LD_LIBRARY_PATH:+:${LD_LIBRARY_PATH}}
# 或者使用下面的命令设置CUDA路径
export CUDA_HOME="/usr/local/cuda-12.4"
export PATH="$CUDA_HOME/bin:$PATH"
export LD_LIBRARY_PATH="$CUDA_HOME/lib64:$LD_LIBRARY_PATH"
# 设置Hydra调试模式（可选）
export HYDRA_FULL_ERROR=1

In [ ]:
# 当打开.bashrc文件是空白的，可以使用如下命令解决：
sudo ls -al ~/ | grep bashrc <br>
sudo vim ~/.bashrc <br>
source ~/.bashrc <br>

##### 切换 CUDA 版本

In [ ]:
# 切换到CUDA 12.4
sudo ln -sf /usr/local/cuda-12.4 /usr/local/cuda
export CUDA_HOME=/usr/local/cuda-12.4
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH

##### 查看指向路径

In [ ]:
echo $LD_LIBRARY_PATH

#### cuDNN安装

In [ ]:
# 查看是否安装cuDNN
# 通用方法（注意路径可能随版本变化）
# （/usr/local/cuda-12.4是指向的CUDA路径）
   cat /usr/local/cuda-12.4/include/cudnn_version.h | grep CUDNN_MAJOR -A 2  # 适用于较新版本 [[9,15]]
   # 或传统方法
   cat /usr/local/cuda-12.4/include/cudnn.h | grep CUDNN_MAJOR -A 2          # 旧版本 [[8,15]]

In [ ]:
wget https://developer.download.nvidia.com/compute/cudnn/redist/cudnn/linux-x86_64/cudnn-linux-x86_64-9.7.1.26_cuda12-archive.tar.xz
xz -dk cudnn-linux-x86_64-9.7.1.26_cuda12-archive.tar.xz
tar -xvf cudnn-linux-x86_64-9.7.1.26_cuda12-archive.tar
cd cudnn-linux-x86_64-9.7.1.26_cuda12-archive
sudo cp ./lib/* /usr/local/cuda-12.4/lib/
sudo cp ./include/* /usr/local/cuda-12.4/include/